# Native Oracle Workshop

Preserve all19 source code stages and original register sizes. Exercise placeholders become labeled executable reference solutions. Quantum functions express predicate XOR, auxiliary phase kickback, direct conditional phase,Deutsch-Jozsa,and Grover search. The two fixed source callback applications are explicit native helpers; this does not claim a generic Python phase_oracle/grover_search/qperm/QStruct API. Named native x/y ports replace source struct fields. Native constant phase is phase(1,pi).


In [ ]:
import math
from typing import Annotated

from examples.synthesis.tutorials.workshops.oracle_workshop.oracles_workshop.overview.workflow import (
    PhaseRow,
    StateCheck,
    classify_deutsch_jozsa,
    phase_table,
    sample,
    verify_program,
)
from flagquantum import synth
from flagquantum.synth import (
    H,
    Output,
    QBit,
    QNum,
    X,
    allocate,
    apply,
    control,
    drop,
    hadamard_transform,
    phase,
    qfunc,
    within,
)

checks: list[StateCheck] = []
shot_counts: list[int] = []
phase_tables: list[tuple[PhaseRow, ...]] = []


def check_program(name: str, program: synth.Program) -> None:
    row = verify_program(name, program)
    checks.append(row)
    print(name, "Full-state error:", row.maximum_state_error)
    print("Dirty hidden workspace:", row.dirty_workspace_probability)
    print(program.report["resources"])


## Reversible predicate oracle

The target is XORed with the Boolean predicate, so a nonzero target is meaningful. x has one bit and y has two. The condition has one solution in this domain:(1,2).


In [ ]:
@qfunc
def oracle(x: QNum, y: QNum, z: QBit) -> None:
    z ^= 2 * x + y == 4


## Prepare the original uniform inputs


In [ ]:
@qfunc
def bit_oracle(
    x: Output[Annotated[QNum, 1]],
    y: Output[Annotated[QNum, 2]],
    z: Output[QBit],
) -> None:
    allocate(x)
    allocate(y)
    allocate(z)
    hadamard_transform(x)
    hadamard_transform(y)
    oracle(x, y, z)


## Synthesize and inspect resources

Public resource reports replace the hosted circuit viewer. Counts are synthesized instructions before Runtime compilation,not hardware-basis counts.


In [ ]:
bit_program = synth.synthesize(bit_oracle, memory_limit_bytes=64 * 1024**2)
check_program("bit_oracle", bit_program)


## Phase kickback exercise reference solutions

First create an auxiliary in|-> using X then H,apply the predicate XOR,and reverse preparation through within/apply. That returns the auxiliary to zero. drop then ends its lifetime; drop is not a reset. Compare with direct predicate-controlled phase. Preserve both executable methods rather than replacing the auxiliary lesson.


In [ ]:
@qfunc
def auxiliary_kickback(
    x: Output[Annotated[QNum, 1]], y: Output[Annotated[QNum, 2]]
) -> None:
    allocate(x)
    allocate(y)
    hadamard_transform(x)
    hadamard_transform(y)
    z: QBit = QBit()
    allocate(z)
    with within():
        X(z)
        H(z)
    with apply():
        oracle(x, y, z)
    drop(z)


@qfunc
def direct_kickback(
    x: Output[Annotated[QNum, 1]], y: Output[Annotated[QNum, 2]]
) -> None:
    allocate(x)
    allocate(y)
    hadamard_transform(x)
    hadamard_transform(y)
    control(2 * x + y == 4, lambda: phase(1, math.pi))


auxiliary_program = synth.synthesize(
    auxiliary_kickback, memory_limit_bytes=64 * 1024**2
)
direct_program = synth.synthesize(direct_kickback, memory_limit_bytes=64 * 1024**2)
check_program("auxiliary_kickback", auxiliary_program)
check_program("direct_kickback", direct_program)


## Inspect actual complex phases

Retain the source magnitude/angle-in-pi table and identify the marked equation solution. Independent checks include global phase; no phase alignment is used.


In [ ]:
for program in (auxiliary_program, direct_program):
    rows = phase_table(program)
    phase_tables.append(rows)
    print("x y |amplitude| angle/pi solution amplitude")
    for row in rows:
        print(row)


## Deutsch-Jozsa exercise reference solution

The three-bit function is x%2==0. A within/apply Hadamard pair converts its phase into a deterministic nonzero readout.


In [ ]:
@qfunc
def deutsch_jozsa(x: Output[Annotated[QNum, 3]]) -> None:
    allocate(x)
    with within():
        hadamard_transform(x)
    with apply():
        control(x % 2 == 0, lambda: phase(1, math.pi))


In [ ]:
deutsch_program = synth.synthesize(deutsch_jozsa, memory_limit_bytes=64 * 1024**2)
check_program("deutsch_jozsa", deutsch_program)


## Actual readout and classification

The source sample default is made explicit as1000 actual local draws. The classical classifier uses observed nonzero-count words. It reports undetermined for multiple observed outputs rather than inventing a promise result.


In [ ]:
deutsch_observations = sample(deutsch_program, shots=1000, seed=1970)
shot_counts.append(sum(row.count for row in deutsch_observations))
print(deutsch_observations)
print("The function is", classify_deutsch_jozsa(deutsch_observations))


## Grover predicate exercise reference solution

Two unsigned two-bit words use the integer predicate x-y==2,including negative differences. The helper XORs a borrowed mark bit. This is a fixed native equivalent of the source qperm operation; no generic qperm decorator is asserted.


In [ ]:
@qfunc
def difference_oracle(x: QNum, y: QNum, z: QBit) -> None:
    z ^= x - y == 2


## Two-round native Grover composition

Preserve the source solution count and reps=2, replacing the exercise placeholder5. The root explicitly owns the auxiliary; the phase helper borrows it and returns it to zero on each call. Diffusion includes the global minus required for2|s><s|-I. The fixed source phase_oracle/grover_search applications are expanded into visible native helpers with the same algorithm.


In [ ]:
@qfunc
def difference_phase(x: QNum, y: QNum, z: QBit) -> None:
    with within():
        X(z)
        H(z)
    with apply():
        difference_oracle(x, y, z)


@qfunc
def diffuse(x: QNum, y: QNum) -> None:
    hadamard_transform(x)
    hadamard_transform(y)
    control((x == 0) & (y == 0), lambda: phase(1, math.pi))
    phase(1, math.pi)
    hadamard_transform(x)
    hadamard_transform(y)


@qfunc
def grover(x: Output[Annotated[QNum, 2]], y: Output[Annotated[QNum, 2]]) -> None:
    allocate(x)
    allocate(y)
    hadamard_transform(x)
    hadamard_transform(y)
    z: QBit = QBit()
    allocate(z)
    for _ in range(2):
        difference_phase(x, y, z)
        diffuse(x, y)
    drop(z)


In [ ]:
grover_program = synth.synthesize(grover, memory_limit_bytes=64 * 1024**2)
check_program("grover", grover_program)
grover_observations = sample(grover_program, shots=1000, seed=1971)
shot_counts.append(sum(row.count for row in grover_observations))
print(grover_observations)


## Phase-kickback solution replay

Rebuild and execute the direct native solution from the source solution section.


In [ ]:
direct_solution = synth.synthesize(direct_kickback, memory_limit_bytes=64 * 1024**2)
check_program("direct_kickback", direct_solution)


In [ ]:
solution_rows = phase_table(direct_solution)
phase_tables.append(solution_rows)
for row in solution_rows:
    print(row)


## Deutsch-Jozsa solution replay


In [ ]:
deutsch_solution = synth.synthesize(deutsch_jozsa, memory_limit_bytes=64 * 1024**2)
check_program("deutsch_jozsa", deutsch_solution)


In [ ]:
deutsch_solution_observations = sample(deutsch_solution, shots=1000, seed=1972)
shot_counts.append(sum(row.count for row in deutsch_solution_observations))
print(deutsch_solution_observations)
print("The function is", classify_deutsch_jozsa(deutsch_solution_observations))


## Grover solution replay

Reuse the same checked XOR helper and original two-bit fields. These references introduce no new algorithm.


In [ ]:
print("Native difference oracle:", difference_oracle)


In [ ]:
grover_solution = synth.synthesize(grover, memory_limit_bytes=64 * 1024**2)


In [ ]:
check_program("grover", grover_solution)
grover_solution_observations = sample(grover_solution, shots=1000, seed=1973)
shot_counts.append(sum(row.count for row in grover_solution_observations))
print(grover_solution_observations)


All programs execute locally without provider credentials or Qmod source imports. No hardware submission,optimizer/scalability advantage,resource optimality or speedup is claimed. Auxiliary and compiler cleanup is checked from actual full complex states; drop itself does not erase entanglement.
